# LSTM Text Prediction System
Predict next word using LSTM


In [11]:
!pip install tensorflow

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import requests


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


2. Dataset (Project Gutenberg)

In [14]:
# Dataset Collection from Open Dataset (Project Gutenberg)
url = "https://www.gutenberg.org/files/11/11-0.txt"  # Alice's Adventures in Wonderland
response = requests.get(url)
full_text = response.text

# Clean and limit the data
data = full_text[1000:15000]  # Skip header, take a portion
data = data.replace('\r\n', ' ').replace('\n', ' ')  # Clean newlines
data = [sentence.strip() for sentence in data.split('.') if sentence.strip()]  # Split into sentences

3. Preprocessing

## Dataset Declaration

**Dataset Name:** Alice's Adventures in Wonderland (excerpt)

**Source:** Project Gutenberg (https://www.gutenberg.org/files/11/11-0.txt)

**Description:** A portion of the classic novel by Lewis Carroll, used for text prediction training. Contains narrative text with vocabulary suitable for sequence learning.

**Preprocessing Steps:**
- Fetched raw text from URL
- Removed header/footer content
- Cleaned newline characters
- Limited to 15,000 characters for manageable training

In [16]:
# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)

total_words = len(tokenizer.word_index) + 1

# Create sequences
input_sequences = []

for line in data:
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

# Padding
max_len = max(len(seq) for seq in input_sequences)

input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_len, padding='pre'))

# Split into X and y
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encoding
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

4. Build LSTM Model

In [17]:
model = Sequential()

model.add(Embedding(total_words, 100, input_length=max_len-1))
model.add(LSTM(150))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### LSTM Model Explanation

- **Input Gate:** Determines which new information enters the cell state.
- **Forget Gate:** Controls what information is kept or discarded from the previous cell state.
- **Output Gate:** Decides what part of the cell state is used to produce the hidden state.
- **Cell State:** Carries long-term memory across time steps.
- **Hidden State:** Represents the current output of the LSTM layer.

LSTM learns sequence patterns by updating the cell and hidden states at each time step, allowing the model to remember important context for next-word prediction.

5. Train Model

In [18]:
model.fit(X, y, epochs=100, verbose=1)

Epoch 1/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 26s 183ms/step - accuracy: 0.0308 - loss: 6.0941
Epoch 2/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 15s 179ms/step - accuracy: 0.0434 - loss: 5.6981
Epoch 3/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 14s 169ms/step - accuracy: 0.0434 - loss: 5.5845
Epoch 4/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 15s 176ms/step - accuracy: 0.0468 - loss: 5.4891
Epoch 5/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 20s 235ms/step - accuracy: 0.0529 - loss: 5.4021
Epoch 6/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 24s 292ms/step - accuracy: 0.0731 - loss: 5.2910
Epoch 7/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 26s 307ms/step - accuracy: 0.0876 - loss: 5.1272
Epoch 8/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 26s 311ms/step - accuracy: 0.1150 - loss: 4.9332
Epoch 9/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 26s 310ms/step - accuracy: 0.1287 - loss: 4.7153
Epoch 10/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 29s 348ms/step - accuracy: 0.1538 - loss: 4.4853
Epoch 11/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 31s 372ms/step - accuracy: 0.1736 - loss: 4.2814
Epoch 12/100
83/83 ━━━━━━━━━━━

In [19]:
def predict_next_word(text):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
    
    predicted = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted)
    
    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word

7. Test Prediction

In [20]:
print(predict_next_word("I love"))
print(predict_next_word("machine learning"))

if
miss


8. Save Model & Tokenizer

In [9]:
model.save("model.h5")

import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save max_len
with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)

In [ ]:
# Test the API (assuming server is running on localhost:8000)
import requests

# Test predictions
test_inputs = ["I love", "machine learning", "deep learning"]

for text in test_inputs:
    response = requests.post("http://127.0.0.1:8000/predict", json={"text": text})
    if response.status_code == 200:
        result = response.json()
        print(f"Input: '{result['input']}' -> Next Word: '{result['next_word']}'")
    else:
        print(f"Error for '{text}': {response.status_code}")

### Deployment and API Testing

- Start the FastAPI server with:
  ```bash
  uvicorn main:app --reload
  ```
- Use the web UI in `index.html` or send POST requests to `/predict`.
- Example request body: `{"text": "I love"}`
- Example response:
  ```json
  {
    "input": "I love",
    "next_word": "artificial"
  }
  ```

> Make sure the FastAPI app is running before executing the API test cell.

In [ ]:
import os
import shutil
import subprocess

workspace = r"C:\Users\Chetanraje\Downloads\DL_Assignmnet_5"
clone_dir = r"C:\Users\Chetanraje\Downloads\temp_upload_repo3"
remote = "https://github.com/chetanraje27/Deep-Learning-Assignments.git"

# Clone remote repo
if os.path.exists(clone_dir):
    shutil.rmtree(clone_dir)
subprocess.run(["git", "clone", remote, clone_dir], check=True)

# Ensure target folder exists
assignment_dir = os.path.join(clone_dir, "Assignment_No_5_LSTM")
os.makedirs(assignment_dir, exist_ok=True)

# Copy files from workspace into repository folder
for name in os.listdir(workspace):
    src = os.path.join(workspace, name)
    if os.path.isfile(src):
        shutil.copy2(src, assignment_dir)

# Commit and push
subprocess.run(["git", "-C", clone_dir, "add", "Assignment_No_5_LSTM"], check=True)
subprocess.run(["git", "-C", clone_dir, "commit", "-m", "Upload Assignment No. 5 LSTM project"], check=True)
subprocess.run(["git", "-C", clone_dir, "push", "origin", "main"], check=True)
print("Upload complete")

PermissionError: [WinError 5] Access is denied: 'C:\\Users\\Chetanraje\\Downloads\\temp_upload_repo2\\.git\\objects\\pack\\pack-176fcfffbf0719aae9ca44a2832db27c61c17fc6.idx'